In [1]:
import numpy as np
import pandas as pd
from db_utils import db_conectar


In [2]:
conn = db_conectar()

In [3]:
query = "SELECT * FROM Silver.TBL_ENCUESTADOS_SILVER"
df = pd.read_sql(query, conn)

C:\Users\54112\AppData\Local\Temp\ipykernel_34232\1096562403.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Al tener una encuesta de 2023 lo que vamos a transformar son los salarios y enmascararlos para mantener la privacidad
¿Como vamos a actualizar los salarios si son de 2023?
Primero a los salarios netos que ingresaron los encuestados le vamos a aplicar la inflacion que hubo de 2023 hasta la actualidad
¿Y si el encuestado cobraba en dolares?
Lo que se va hacer es tomar su sueldo en dolares y SI INGRESO el valor dolar en su momento hacemos:
sueldo_dolar x valor_dolar =  salario_neto_ars 
A salario_neto_Ars le aplicamos la inflacion y obtenemos el sueldo actualizado
Pero ¿Y si solo puso su salario en dolares y no puso valor dolar?
Para esto encontramos la solucion de tomar el valor medio o promedio del dolar en el año 2023 y adjuntarselo entonce sería:
sueldo_dolar x media_dolar = salario_neto_ars
Y ahí se le aplica nuevamente la inflacion para obtener el salario actual.


Declaramos variables

In [ ]:
#Tomamos los datos
DOLAR_PROMEDIO_2023 = 306.7
INFLACION_2023 = 2.114
INFLACION_2024 = 1.178
INFLACION_2025_ACUM = 0.151

# Calculamos la inflacion total de 2023 a 2025
CALCULO_INFLACION_2023_2025 = (1 + INFLACION_2023) * (1 + INFLACION_2024) * (1 + INFLACION_2025_ACUM)
print("Inflación acumulada 2023-2025:", CALCULO_INFLACION_2023_2025)

Inflación acumulada 2023-2025: 7.8064180919999995


In [ ]:
#Copiamos el dataframe de silver que es directamente la tabla encuestados
df_gold = df.copy()


In [6]:
df_gold["valor_dolar"] = pd.to_numeric(df_gold["valor_dolar"], errors="coerce")

df_gold["salario_neto_ars"] = (
    df_gold["salario_neto_ars"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [7]:
df_gold.loc[
    (df_gold["valor_dolar"] < 150) | (df_gold["valor_dolar"] > 1000),
    "valor_dolar"
] = np.nan


In [ ]:
# Sueldos en pesos
mask_pesos = df_gold["id_sueldo_dolarizado"] == 2
df_gold.loc[mask_pesos, "salario_real"] = (
    df_gold.loc[mask_pesos, "salario_neto_ars"] * CALCULO_INFLACION_2023_2025
)

#Sueldo en dolares con valor_dolar
mask_usd_valor = (df_gold["id_sueldo_dolarizado"] == 1) & (df_gold["valor_dolar"].notna())
df_gold.loc[mask_usd_valor, "salario_real"] = (
    df_gold.loc[mask_usd_valor, "salario_neto_ars"] *
    df_gold.loc[mask_usd_valor, "valor_dolar"] *
    CALCULO_INFLACION_2023_2025
)

# Sueldo en dolares sin valor_dolar, usamos promedio de 2023
mask_usd_sin_valor = (df_gold["id_sueldo_dolarizado"] == 1) & (df_gold["valor_dolar"].isna())
df_gold.loc[mask_usd_sin_valor, "salario_real"] = (
    df_gold.loc[mask_usd_sin_valor, "salario_neto_ars"] *
    DOLAR_PROMEDIO_2023 *
    CALCULO_INFLACION_2023_2025
)

In [ ]:
df_gold = df_gold[df_gold["id_jornada"] == 1]

In [10]:
def filtrar_por_cuantiles(grupo):
    q1 = grupo["salario_real"].quantile(0.01)
    q99 = grupo["salario_real"].quantile(0.99)
    return grupo[(grupo["salario_real"] >= q1) & (grupo["salario_real"] <= q99)]

df_gold = df_gold.groupby("id_seniority", group_keys=False).apply(filtrar_por_cuantiles)

C:\Users\54112\AppData\Local\Temp\ipykernel_34232\3942859426.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_gold = df_gold.groupby("id_seniority", group_keys=False).apply(filtrar_por_cuantiles)


Enmascaramos el salario para evitar que se sepan los datos reales

In [11]:
rangos_salario = pd.DataFrame([
    {"id_rango_salario": 1, "salario_min": 0,        "salario_max": 500000,   "descripcion": "0 - 500K"},
    {"id_rango_salario": 2, "salario_min": 500000,   "salario_max": 1000000,  "descripcion": "500K - 1M"},
    {"id_rango_salario": 3, "salario_min": 1000000,  "salario_max": 1500000,  "descripcion": "1M - 1.5M"},
    {"id_rango_salario": 4, "salario_min": 1500000,  "salario_max": 2500000,  "descripcion": "1.5M - 2.5M"},
    {"id_rango_salario": 5, "salario_min": 2500000,  "salario_max": 4000000,  "descripcion": "2.5M - 4M"},
    {"id_rango_salario": 6, "salario_min": 4000000,  "salario_max": 7000000,  "descripcion": "4M - 7M"},
    {"id_rango_salario": 7, "salario_min": 7000000,  "salario_max": 15000000, "descripcion": "7M - 15M"},
])


In [12]:

def asignar_rango_salario(valor):
    if pd.isnull(valor):
        return None
    for _, r in rangos_salario.iterrows():
        if r["salario_max"] is None:
            if valor >= r["salario_min"]:
                return r["id_rango_salario"]
        elif r["salario_min"] <= valor < r["salario_max"]:
            return r["id_rango_salario"]
    return None

df_gold["id_rango_salario"] = df_gold["salario_real"].apply(asignar_rango_salario)

In [ ]:
cursor = conn.cursor()
cursor.execute("""
IF OBJECT_ID('Gold.TBL_ENCUESTADOS_GOLD', 'U') IS NOT NULL
    DROP TABLE Gold.TBL_ENCUESTADOS_GOLD;
""")


cursor.execute("""
IF OBJECT_ID('Gold.TBL_RANGO_SALARIO', 'U') IS NOT NULL
    DROP TABLE Gold.TBL_RANGO_SALARIO;
""")

cursor.execute("""
CREATE TABLE Gold.TBL_RANGO_SALARIO (
    ID_RANGO_SALARIO INT PRIMARY KEY,
    SALARIO_MIN NUMERIC(18,2) NULL,
    SALARIO_MAX NUMERIC(18,2) NULL,
    DESCRIPCION NVARCHAR(50) NOT NULL
);
""")

for i, row in rangos_salario.iterrows():
    salario_min = None if pd.isnull(row["salario_min"]) else float(row["salario_min"])
    salario_max = None if pd.isnull(row["salario_max"]) else float(row["salario_max"])

    cursor.execute("""
    INSERT INTO Gold.TBL_RANGO_SALARIO (ID_RANGO_SALARIO, SALARIO_MIN, SALARIO_MAX, DESCRIPCION)
    VALUES (?, ?, ?, ?)
    """, int(row["id_rango_salario"]), salario_min, salario_max, row["descripcion"])

conn.commit()

Como ya tenemos el enmascaramiento de el salario pasamos a crear la tabla encuestados en GOLD

In [14]:
cursor.execute("""
CREATE TABLE Gold.TBL_ENCUESTADOS_GOLD (
    ID_ENCUESTADO INT PRIMARY KEY,
    EDAD INT NULL,
    ID_GENERO INT NULL,
    ID_PROVINCIA INT NULL,
    ID_SENIORITY INT NULL,
    ID_TIPO_CONTRATO INT NULL,
    ID_MODALIDAD INT NULL,
    ID_JORNADA INT NULL,
    ANIOS_EXPERIENCIA INT NULL,
    ANTIGUEDAD_EMPRESA INT NULL,
    TIEMPO_PUESTO INT NULL,
    ID_NIVEL_ESTUDIOS INT NULL,
    SALARIO_REAL NUMERIC(18,2) NULL,
    ID_RANGO_SALARIO INT NULL,
    ID_SUELDO_DOLARIZADO INT NULL,
    ID_BONO INT NULL,
    ID_ACTUALIZACION_SALARIAL INT NULL,
    ID_GUARDIAS INT NULL,
    ID_SATISFACCION_INGRESOS INT NULL,
    ID_BUSCANDO_TRABAJO INT NULL,
    FOREIGN KEY (ID_GENERO) REFERENCES Silver.TBL_GENERO(ID),
    FOREIGN KEY (ID_PROVINCIA) REFERENCES Silver.TBL_PROVINCIA(ID),
    FOREIGN KEY (ID_SENIORITY) REFERENCES Silver.TBL_SENIORITY(ID),
    FOREIGN KEY (ID_TIPO_CONTRATO) REFERENCES Silver.TBL_CONTRATO(ID),
    FOREIGN KEY (ID_MODALIDAD) REFERENCES Silver.TBL_MODALIDAD(ID),
    FOREIGN KEY (ID_JORNADA) REFERENCES Silver.TBL_JORNADA(ID),
    FOREIGN KEY (ID_NIVEL_ESTUDIOS) REFERENCES Silver.TBL_ESTUDIOS(ID),
    FOREIGN KEY (ID_SUELDO_DOLARIZADO) REFERENCES Silver.TBL_SUELDO_DOLARIZADO(ID),
    FOREIGN KEY (ID_BONO) REFERENCES Silver.TBL_BONO(ID),
    FOREIGN KEY (ID_ACTUALIZACION_SALARIAL) REFERENCES Silver.TBL_ACTUALIZACION_SALARIAL(ID),
    FOREIGN KEY (ID_GUARDIAS) REFERENCES Silver.TBL_GUARDIAS(ID),
    FOREIGN KEY (ID_SATISFACCION_INGRESOS) REFERENCES Silver.TBL_SATISFACCION_INGRESOS(ID),
    FOREIGN KEY (ID_BUSCANDO_TRABAJO) REFERENCES Silver.TBL_BUSCANDO_TRABAJO(ID),
    FOREIGN KEY (ID_RANGO_SALARIO) REFERENCES Gold.TBL_RANGO_SALARIO(ID_RANGO_SALARIO)
);
""")

conn.commit()

In [ ]:
data = [
    (
        int(row["ID_ENCUESTADO"]),
        int(row["edad"]) if not pd.isnull(row["edad"]) else None,
        int(row["id_genero"]) if not pd.isnull(row["id_genero"]) else None,
        int(row["id_provincia"]) if not pd.isnull(row["id_provincia"]) else None,
        int(row["id_seniority"]) if not pd.isnull(row["id_seniority"]) else None,
        int(row["id_tipo_contrato"]) if not pd.isnull(row["id_tipo_contrato"]) else None,
        int(row["id_modalidad"]) if not pd.isnull(row["id_modalidad"]) else None,
        int(row["id_jornada"]) if not pd.isnull(row["id_jornada"]) else None,
        int(row["anios_experiencia"]) if not pd.isnull(row["anios_experiencia"]) else None,
        int(row["antiguedad_empresa"]) if not pd.isnull(row["antiguedad_empresa"]) else None,
        int(row["tiempo_puesto"]) if not pd.isnull(row["tiempo_puesto"]) else None,
        int(row["id_nivel_estudios"]) if not pd.isnull(row["id_nivel_estudios"]) else None,
        float(row["salario_real"]) if not pd.isnull(row["salario_real"]) else None,
        int(row["id_rango_salario"]) if not pd.isnull(row["id_rango_salario"]) else None,
        int(row["id_sueldo_dolarizado"]) if not pd.isnull(row["id_sueldo_dolarizado"]) else None,
        int(row["id_bono"]) if not pd.isnull(row["id_bono"]) else None,
        int(row["id_actualizacion_salarial"]) if not pd.isnull(row["id_actualizacion_salarial"]) else None,
        int(row["id_guardias"]) if not pd.isnull(row["id_guardias"]) else None,
        int(row["id_satisfaccion_ingresos"]) if not pd.isnull(row["id_satisfaccion_ingresos"]) else None,
        int(row["id_buscando_trabajo"]) if not pd.isnull(row["id_buscando_trabajo"]) else None,
    )
    for _, row in df_gold.iterrows()
]

print(f"Insertando {len(data)} registros en Gold.TBL_ENCUESTADOS_GOLD...")

# Insertar datos en nuestra tabla de transformacion gold
cursor.fast_executemany = True
cursor.executemany("""
INSERT INTO Gold.TBL_ENCUESTADOS_GOLD (
    ID_ENCUESTADO, EDAD, ID_GENERO, ID_PROVINCIA, ID_SENIORITY, ID_TIPO_CONTRATO, 
    ID_MODALIDAD, ID_JORNADA, ANIOS_EXPERIENCIA, ANTIGUEDAD_EMPRESA, TIEMPO_PUESTO, 
    ID_NIVEL_ESTUDIOS, SALARIO_REAL, ID_RANGO_SALARIO, ID_SUELDO_DOLARIZADO,
    ID_BONO, ID_ACTUALIZACION_SALARIAL, ID_GUARDIAS, ID_SATISFACCION_INGRESOS, ID_BUSCANDO_TRABAJO
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", data)

conn.commit()